# UNSW-NB15 Preprocessing Notebook
This notebook preprocesses UNSW-NB15 data into a model-ready format.

## Pipeline summary
1. Load raw UNSW training data.
2. Inspect schema and **multi-class** distribution (`attack_cat`).
3. Drop non-informative columns (`id`, `label`) and duplicate rows.
4. Encode categorical features (`proto`, `service`, `state`).
5. Encode target column (`attack_cat`) with LabelEncoder.
6. Normalize numeric **features only** (not the target).
7. Perform train/test split with stratification.
8. Visualize class balance and correlations.
9. Export cleaned dataset to `../data_processed/unsw_nb15_cleaned.csv`.

## Step A: Load and Inspect
These cells load UNSW-NB15 data and inspect schema/summary statistics.
Use this stage to confirm expected columns and **attack category** labels before transformations.

In [ ]:
import pandas as pd

df = pd.read_csv("../data_raw/unsw_nb15/UNSW_NB15_training-set.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Multi-class distribution based on attack_cat
# Strip whitespace from attack_cat values first (raw data sometimes has leading/trailing spaces)
df['attack_cat'] = df['attack_cat'].str.strip()

print("=== Attack Category Distribution ===")
print(df['attack_cat'].value_counts())
print(f"\nTotal categories: {df['attack_cat'].nunique()}")
print(f"Unique categories: {sorted(df['attack_cat'].unique())}")

## Step B: Clean and Transform
This stage drops non-informative columns, encodes categorical features, and removes duplicates.

**Key change**: We drop both `id` and `label` columns. The `label` column (binary 0/1) is no longer needed because we use `attack_cat` for multi-class classification.

In [ ]:
# Drop non-informative columns: id and binary label
df = df.drop(columns=["id", "label"])
print(f"Columns after dropping id and label: {list(df.columns)}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical FEATURES (not the target)
le_proto = LabelEncoder()
le_service = LabelEncoder()
le_state = LabelEncoder()

df["proto"] = le_proto.fit_transform(df["proto"])
df["service"] = le_service.fit_transform(df["service"])
df["state"] = le_state.fit_transform(df["state"])

print("Categorical features encoded: proto, service, state")

In [ ]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows. Remaining: {after}")

## Step C: Encode Target and Scale Features
1. Encode `attack_cat` into integer labels using `LabelEncoder`.
2. Apply `MinMaxScaler` only to feature columns (not the target).

**Important**: The target column should never be scaled.

In [ ]:
# Encode the target column: attack_cat
le_target = LabelEncoder()
df["attack_cat"] = le_target.fit_transform(df["attack_cat"])

# Print the encoding mapping for reference
print("=== attack_cat Encoding Mapping ===")
for i, cls in enumerate(le_target.classes_):
    count = (df['attack_cat'] == i).sum()
    print(f"  {i} -> {cls} ({count} samples)")
print(f"\nTotal classes: {len(le_target.classes_)}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Scale features only (exclude attack_cat target column)
feature_cols = [c for c in df.columns if c != "attack_cat"]
scaler = MinMaxScaler()

df[feature_cols] = scaler.fit_transform(df[feature_cols])
print(f"Scaled {len(feature_cols)} feature columns to [0, 1] range.")
print(f"Feature columns: {feature_cols}")

## Step D: Train/Test Split
Split with `stratify=y` to preserve multi-class proportions across train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("attack_cat", axis=1)
y = df["attack_cat"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest set class distribution:")
print(y_test.value_counts().sort_index())

## Step E: Visualize Class Balance and Correlations
Plots and class reports showing multi-class imbalance, which is critical for IDS datasets.

In [ ]:
import matplotlib.pyplot as plt

# Multi-class distribution bar chart
class_counts = df["attack_cat"].value_counts().sort_index()
class_labels = [le_target.classes_[i] for i in class_counts.index]

plt.figure(figsize=(12, 6))
bars = plt.bar(class_labels, class_counts.values, color='steelblue', edgecolor='black')
plt.title("Multi-Class Distribution - UNSW NB15 (attack_cat)", fontsize=14)
plt.xlabel("Attack Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add count labels on top of bars
for bar, count in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{count}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Multi-class imbalance report
import os

class_counts = df["attack_cat"].value_counts().sort_values(ascending=False)
total_samples = class_counts.sum()
minority_class_idx = class_counts.idxmin()
majority_class_idx = class_counts.idxmax()
minority_count = class_counts.min()
majority_count = class_counts.max()

minority_percentage = (minority_count / total_samples) * 100
imbalance_ratio = majority_count / minority_count

print(f"Minority class: {le_target.classes_[minority_class_idx]} (encoded={minority_class_idx})")
print(f"Minority class count: {minority_count}")
print(f"Minority class percentage: {minority_percentage:.4f}%")
print(f"\nMajority class: {le_target.classes_[majority_class_idx]} (encoded={majority_class_idx})")
print(f"Majority class count: {majority_count}")
print(f"\nImbalance ratio (majority/minority): {imbalance_ratio:.4f}")

print("\n=== Per-Class Report ===")
class_report = pd.DataFrame({
    'class_name': [le_target.classes_[i] for i in class_counts.index],
    'encoded': class_counts.index,
    'count': class_counts.values,
    'percentage': (class_counts.values / total_samples * 100).round(4)
})
class_report

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr())
plt.title("Correlation Heatmap")
plt.show()

## Step F: Export Cleaned Dataset
The exported CSV uses integer-encoded `attack_cat` as the target column.

Downstream notebooks should use `attack_cat` as the target column `y`.

In [ ]:
df.to_csv("../data_processed/unsw_nb15_cleaned.csv", index=False)
print(f"Saved cleaned dataset to ../data_processed/unsw_nb15_cleaned.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget column: attack_cat")
print(f"Number of classes: {df['attack_cat'].nunique()}")